# 08 — Consolidate the LLM metadata and build the API database

Run from the `notebooks/` folder. Needs only `pandas`.

Every published field keeps its **source** (`llm`, `llm+fasttext`, `regex`, `pdf_embedded`, `fasttext`) and a **confidence** (`high`, `medium`, `low`), so API users can filter by trust. Nothing here changes the earlier outputs.

**Before this notebook:** run notebook 07 with `RUN_SCOPE = 'all'` so every document has an LLM answer. Documents without one are still published, from fallbacks only, and flagged `no_llm_output`.

## 1. Settings and inputs

In [3]:
from pathlib import Path
import json, re, os, sqlite3, unicodedata, difflib, datetime
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / 'output/inventory/documents.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the project root or the notebooks folder, after notebooks 01-07.')
INVENTORY_DIR = ROOT / 'output/inventory'
PREPROC_DIR = ROOT / 'output/preprocessing'
EXTRACT_DIR = ROOT / 'output/extraction'
API_DIR = ROOT / 'output/api'
API_DIR.mkdir(parents=True, exist_ok=True)

PREFERRED_MODEL = None      # None = newest answer per document; or e.g. 'Qwen/Qwen2.5-1.5B-Instruct'
KEEP_UNGROUNDED = False     # True publishes LLM values that are NOT found in the page text (confidence 'low')
GROUNDING_PAGES = 4         # same pages the extraction prompt saw (notebook 07 reads up to 4)
MIN_TITLE_COVERAGE = 0.90   # share of the title that must be found in the page text
EVAL_SPLITS = ['dev']       # do not add 'test' until you are done tuning
DOC_TYPES = ['journal_article', 'report', 'thesis', 'conference_paper', 'book_chapter', 'policy_brief', 'other']
CURRENT_YEAR = datetime.date.today().year
print('Project root:', ROOT)

Project root: /Users/georgyfurs/Documents/ICT_Year4/CPS/DDD


In [4]:
def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

documents = read_csv(INVENTORY_DIR / 'documents.csv')
observations = read_csv(INVENTORY_DIR / 'metadata_observations.csv')
terms = read_csv(INVENTORY_DIR / 'document_terms.csv')

# page text for grounding (written by notebook 05)
pages_path = PREPROC_DIR / 'pages.jsonl'
if not pages_path.is_file():
    raise FileNotFoundError(f'{pages_path} not found. Run notebook 05 first.')
pages = pd.read_json(pages_path, lines=True, convert_dates=False, dtype={'document_id': str})
source_pages = {d: g.sort_values('page_number').head(GROUNDING_PAGES).original_text.tolist()
                for d, g in pages.groupby('document_id')}

# raw LLM answers (written by notebook 07)
llm_raw = {}
raw_path = EXTRACT_DIR / 'eurollm_raw.jsonl'
if raw_path.is_file():
    for line in raw_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if PREFERRED_MODEL and r['model'] != PREFERRED_MODEL:
            continue
        llm_raw[r['document_id']] = (r['raw'], r['model'])   # later lines win
else:
    print('WARNING: no eurollm_raw.jsonl found; everything will come from fallbacks.')

# candidate values from notebook 01
obs_by_doc = {}
for r in observations.itertuples(index=False):
    obs_by_doc.setdefault(r.document_id, {}).setdefault(r.field_name, []).append(r.observed_value)
emb_keywords = {}
for r in terms[(terms.term_type == 'keyword') & (terms.source_type == 'pdf_embedded')].itertuples(index=False):
    emb_keywords.setdefault(r.document_id, []).append(r.term_value)

# fastText shares per document (notebooks 05)
lang_share = {}
sec_path, lang_path = PREPROC_DIR / 'sections.jsonl', PREPROC_DIR / 'section_languages.csv'
if sec_path.is_file() and lang_path.is_file():
    sec = pd.read_json(sec_path, lines=True, convert_dates=False,
                       dtype={'document_id': str})[['document_id', 'section_number', 'character_count']]
    sl = pd.read_csv(lang_path, keep_default_na=False)
    j = sl.merge(sec, on=['document_id', 'section_number'])
    ok = j[j.detected_language.isin(['en', 'nl']) & (j.language_score >= 0.80) & (j.character_count >= 100)]
    tot = ok.groupby(['document_id', 'detected_language']).character_count.sum().unstack(fill_value=0)
    lang_share = tot.div(tot.sum(axis=1), axis=0).to_dict('index')

print(f'{len(documents)} documents | {len(llm_raw)} with an LLM answer | '
      f'{len(source_pages)} with page text | {len(lang_share)} with fastText shares')

94 documents | 8 with an LLM answer | 94 with page text | 93 with fastText shares


## 2. Cleaning and grounding helpers

`squash` removes spaces, punctuation, accents and ligatures, so that text from PDFs with missing spaces (e.g. `reliesontheadaptivecapacities`) can still be matched. A value is grounded when its squashed form is found in the squashed page text.

In [6]:
NULL_STRINGS = {'', 'null', 'none', 'n/a', 'na', 'nan', 'undefined', 'unknown', '-', '[]', '{}'}

def squash(s):
    s = unicodedata.normalize('NFKD', str(s)).lower()
    return re.sub(r'[^a-z0-9]+', '', ''.join(c for c in s if not unicodedata.combining(c)))

def is_null(v):
    if v is None:
        return True
    if isinstance(v, str):
        return v.strip().lower() in NULL_STRINGS
    if isinstance(v, (list, dict)):
        return len(v) == 0
    return False

def norm_str(v):
    if is_null(v):
        return None
    s = re.sub(r'\s+', ' ', str(v)).strip()
    return None if s.lower() in NULL_STRINGS else s

def norm_list(v):
    if is_null(v):
        return []
    if isinstance(v, str):
        parts = re.split(r'[;|\n]', v)
        if len(parts) == 1 and re.search(r'\s(?:and|en|&)\s', v):
            parts = re.split(r'\s(?:and|en|&)\s', v)
        v = parts
    elif isinstance(v, dict):
        v = [v]
    items = []
    for x in v:
        if isinstance(x, dict):
            x = (x.get('name') or x.get('full_name')
                 or ' '.join(str(p) for p in (x.get('given'), x.get('family')) if p))
        s = norm_str(x)
        if s and s not in items:
            items.append(s)
    return items

def clean_author(name):
    name = re.sub(r'[\*†‡§\d]+$', '', name).strip(' ,;')
    parts = [p.strip() for p in name.split(',')]
    if (len(parts) == 2 and all(re.search(r'[^\W\d_]', p) for p in parts)
            and not re.fullmatch(r'(?i)(jr|sr|phd|md)\.?', parts[1])):
        name = f'{parts[1]} {parts[0]}'          # "Buijs, Jean-Marie" -> "Jean-Marie Buijs"
    return name or None

DOI_RE = re.compile(r'10\.\d{4,9}/[^\s"<>]+', re.I)
def clean_doi(v):
    s = norm_str(v)
    if not s:
        return None
    s = re.sub(r'(?i)^(https?://(dx\.)?doi\.org/|doi:\s*)', '', s)
    m = DOI_RE.search(s)
    if not m:
        return None
    d = m.group(0).rstrip('.,;]')
    while d.endswith(')') and d.count(')') > d.count('('):
        d = d[:-1]
    return d.lower()

def clean_year(v):
    s = norm_str(v)
    m = re.search(r'\b(1[89]\d{2}|20\d{2})\b', s) if s else None
    y = int(m.group(1)) if m else None
    return y if y and 1900 <= y <= CURRENT_YEAR + 1 else None

TYPE_ALIASES = {'article': 'journal_article', 'journal': 'journal_article', 'artikel': 'journal_article',
                'paper': 'conference_paper', 'proceedings': 'conference_paper',
                'chapter': 'book_chapter', 'rapport': 'report', 'dissertation': 'thesis',
                'proefschrift': 'thesis', 'master_thesis': 'thesis', 'bachelor_thesis': 'thesis',
                'brief': 'policy_brief'}
def norm_type(v):
    s = norm_str(v)
    if not s:
        return None
    s = re.sub(r'[\s\-]+', '_', s.lower())
    return s if s in DOC_TYPES else TYPE_ALIASES.get(s)

LANG_ALIASES = {'english': 'en', 'en-us': 'en', 'en-gb': 'en', 'eng': 'en',
                'dutch': 'nl', 'nederlands': 'nl', 'nld': 'nl', 'dut': 'nl',
                'mixed': 'mixed', 'bilingual': 'mixed', 'en': 'en', 'nl': 'nl'}
def norm_lang(v):
    s = norm_str(v)
    return LANG_ALIASES.get(s.lower()) if s else None

def coverage(needle, hay_sq):
    n = squash(needle)
    if not n or not hay_sq:
        return 0.0
    if n in hay_sq:
        return 1.0
    m = difflib.SequenceMatcher(None, hay_sq, n, autojunk=False).find_longest_match(0, len(hay_sq), 0, len(n))
    return m.size / len(n)

def author_grounded(name, hay_sq):
    tokens = re.findall(r"[^\W\d_]{2,}", name)
    return bool(tokens) and all(squash(t) in hay_sq for t in tokens)

def junk_title(t):
    low = t.lower()
    return (len(t) < 8 or low.startswith('microsoft word') or low in {'untitled', 'undefined'}
            or low.endswith(('.doc', '.docx', '.pdf', '.indd', '.tex')))

def year_from_text(text):
    tiers = [re.compile(r'(?i)published|available online|gepubliceerd|publicatiedatum'),
             re.compile(r'(?i)©|copyright')]
    for pat in tiers:
        for line in text.splitlines():
            if pat.search(line):
                ys = [int(y) for y in re.findall(r'\b(19\d{2}|20\d{2})\b', line) if int(y) <= CURRENT_YEAR + 1]
                if ys:
                    return max(ys)
    return None

KEYWORD_LINE = re.compile(r'(?im)^\s*(?:key\s*words?|trefwoorden|sleutelwoorden)\s*[:\-–—]\s*(.+)$')
print('helpers ready')

helpers ready


## 3. Parse the raw LLM answers

The model can answer with extra text, code fences, a truncated JSON object (the generation limit is 200 tokens) or `"null"` strings. `parse_llm` returns the fields plus a status: `json` (clean), `salvaged` (fields recovered by pattern), `failed`.

In [8]:
def salvage(text):
    out = {}
    for key in ('title', 'doi', 'document_type', 'language'):
        m = re.search(rf'"{key}"\s*:\s*"((?:[^"\\]|\\.)*)"', text)
        if m:
            try:
                out[key] = json.loads('"' + m.group(1) + '"')
            except json.JSONDecodeError:
                out[key] = m.group(1)
    m = re.search(r'"publication_year"\s*:\s*"?(\d{4})', text)
    if m:
        out['publication_year'] = m.group(1)
    for key in ('authors', 'keywords'):
        m = re.search(rf'"{key}"\s*:\s*\[(.*?)(?:\]|$)', text, re.S)
        if m:
            out[key] = [s for s in re.findall(r'"((?:[^"\\]|\\.)*)"', m.group(1))]
    return out

def parse_llm(raw):
    text = re.sub(r'```(?:json)?', '', raw or '').strip()
    candidates = []
    start = text.find('{')
    if start >= 0:
        candidates.append(text[start:])
    elif text.startswith('"'):
        candidates.append('{' + text)
    for cand in candidates:
        try:
            data, _ = json.JSONDecoder().raw_decode(cand)
            if isinstance(data, dict):
                return data, 'json'
        except json.JSONDecodeError:
            pass
    data = salvage(text)
    return (data, 'salvaged') if data else ({}, 'failed')

llm_by_doc = {}
for d, (raw, model) in llm_raw.items():
    data, status = parse_llm(raw)
    llm_by_doc[d] = (data, status, model)

status_counts = pd.Series([s for _, s, _ in llm_by_doc.values()], dtype=str).value_counts()
print('LLM answers parsed:')
display(status_counts.rename('documents').to_frame())

LLM answers parsed:


,documents
json,5
salvaged,3


## 4. Resolve one document

Rules in order. A field's **confidence** means:

* `high`: LLM value found (exactly) in the page text.
* `medium`: found approximately, partially, or obtained from a regex / PDF-embedded fallback.
* `low`: not verifiable (for example `document_type`, which is not printed in the text) or kept with `KEEP_UNGROUNDED = True`.

`llm_rejected_fields` lists fields where the LLM value was thrown away, which is the most useful number for judging the model.

In [10]:
def R(value, source, confidence):
    return {'value': value, 'source': source, 'confidence': confidence}
EMPTY = R(None, None, None)

TYPE_SIGNALS = {'thesis': r'thesis|proefschrift|scriptie|dissertation', 'report': r'\breport\b|rapport',
                'conference_paper': r'proceedings|conference|congres', 'policy_brief': r'policy brief',
                'book_chapter': r'\bchapter\b|hoofdstuk', 'journal_article': r'\bjournal\b|tijdschrift'}

def fasttext_language(d):
    s = lang_share.get(d)
    if not s:
        return None, 0.0
    en, nl = s.get('en', 0.0), s.get('nl', 0.0)
    if en >= 0.9: return 'en', en
    if nl >= 0.9: return 'nl', nl
    if en >= 0.1 and nl >= 0.1: return 'mixed', max(en, nl)
    return ('en', en) if en >= nl else ('nl', nl)

def resolve(doc):
    d = doc['document_id']
    pgs = source_pages.get(d, [])
    text = '\n'.join(pgs)
    sq = squash(text)
    llm, parse_status, model = llm_by_doc.get(d, ({}, 'no_llm_output', None))
    obs = obs_by_doc.get(d, {})
    rejected, out = [], {}

    # ---- title
    title = EMPTY
    t = norm_str(llm.get('title'))
    if t:
        cov = coverage(t, sq)
        if cov >= 0.999: title = R(t, 'llm', 'high')
        elif cov >= MIN_TITLE_COVERAGE: title = R(t, 'llm', 'medium')
        elif KEEP_UNGROUNDED: title = R(t, 'llm', 'low'); rejected.append('title')
        else: rejected.append('title')
    if title['value'] is None:
        for e in obs.get('title', []):
            e = norm_str(e)
            if e and not junk_title(e) and coverage(e, sq) >= MIN_TITLE_COVERAGE:
                title = R(e, 'pdf_embedded', 'medium'); break
    out['title'] = title

    # ---- authors
    def grade_authors(names, source):
        names = [n for n in (clean_author(x) for x in names) if n]
        good = [n for n in names if author_grounded(n, sq)]
        if names and len(good) == len(names): return R(good, source, 'high'), False
        if good: return R(good, source, 'medium'), True
        if names and KEEP_UNGROUNDED: return R(names, source, 'low'), True
        return EMPTY, bool(names)
    authors, dropped = grade_authors(norm_list(llm.get('authors')), 'llm')
    if dropped: rejected.append('authors')
    if authors['value'] is None:
        authors, _ = grade_authors(norm_list(';'.join(obs.get('author', []))), 'pdf_embedded')
        if authors['value']: authors['confidence'] = 'medium'
    out['authors'] = authors

    # ---- doi
    doi = EMPTY
    x = clean_doi(llm.get('doi'))
    if x:
        if squash(x) in sq: doi = R(x, 'llm', 'high')
        elif KEEP_UNGROUNDED: doi = R(x, 'llm', 'low'); rejected.append('doi')
        else: rejected.append('doi')
    if doi['value'] is None:
        for pg in pgs:                          # page 1 first: later pages often cite other papers
            m = DOI_RE.search(pg)
            if m:
                doi = R(clean_doi(m.group(0)), 'regex', 'medium'); break
    if doi['value'] is None:
        for e in obs.get('doi', []):
            if clean_doi(e):
                doi = R(clean_doi(e), 'regex', 'medium'); break
    out['doi'] = doi

    # ---- publication year
    year = EMPTY
    y, ty = clean_year(llm.get('publication_year')), year_from_text(text)
    if y:
        if ty == y: year = R(y, 'llm', 'high')
        elif str(y) in text: year = R(y, 'llm', 'medium')
        elif KEEP_UNGROUNDED: year = R(y, 'llm', 'low'); rejected.append('publication_year')
        else: rejected.append('publication_year')
    if year['value'] is None and ty:
        year = R(ty, 'regex', 'medium')
    if year['value'] is None:
        for e in obs.get('online_date', []):
            oy = clean_year(e)
            if oy:
                year = R(oy, 'regex', 'medium'); break
    out['publication_year'] = year

    # ---- document type (not verifiable from text alone: corroborate with simple signals)
    dtype = EMPTY
    ty_llm = norm_type(llm.get('document_type'))
    if ty_llm:
        corroborated = bool(re.search(TYPE_SIGNALS.get(ty_llm, r'(?!x)x'), pgs[0] if pgs else '', re.I)
                            or (ty_llm == 'journal_article' and doi['value']))
        dtype = R(ty_llm, 'llm', 'medium' if corroborated else 'low')
    elif not is_null(llm.get('document_type')):
        rejected.append('document_type')
    out['document_type'] = dtype

    # ---- language: LLM, cross-checked with fastText
    lang = EMPTY
    l_llm = norm_lang(llm.get('language'))
    l_ft, strength = fasttext_language(d)
    if l_ft:
        if l_llm == l_ft: lang = R(l_ft, 'llm+fasttext', 'high')
        else:
            lang = R(l_ft, 'fasttext', 'high' if strength >= 0.9 else 'medium')
            if l_llm: rejected.append('language')
    elif l_llm:
        lang = R(l_llm, 'llm', 'low')
    out['language'] = lang

    # ---- keywords
    kws = [k for k in norm_list(llm.get('keywords')) if len(squash(k)) >= 3]
    good = [k for k in kws if squash(k) in sq]
    if good: keywords = R(good, 'llm', 'high' if len(good) == len(kws) else 'medium')
    elif kws and KEEP_UNGROUNDED: keywords = R(kws, 'llm', 'low')
    else: keywords = EMPTY
    if kws and len(good) < len(kws):
        rejected.append('keywords')
    if keywords['value'] is None:
        found = [k for m in KEYWORD_LINE.finditer(text) for k in re.split(r'[;,·•]', m.group(1)) if k.strip()]
        if found: keywords = R([k.strip() for k in found], 'regex', 'medium')
    if keywords['value'] is None and emb_keywords.get(d):
        keywords = R(norm_list(';'.join(emb_keywords[d])), 'pdf_embedded', 'medium')
    out['keywords'] = keywords

    # ---- review flags
    flags = []
    if doc.get('text_status') in ('no_text', 'error'): flags.append('no_text')
    if parse_status == 'no_llm_output': flags.append('no_llm_output')
    elif parse_status == 'failed': flags.append('llm_parse_failed')
    for f in ('title', 'authors', 'publication_year'):
        if out[f]['value'] in (None, []): flags.append(f'no_{f}')
        elif out[f]['confidence'] == 'low': flags.append(f'low_confidence_{f}')
    return out, flags, rejected, parse_status, model

## 5. Run on every document

In [12]:
FIELDS = ['title', 'authors', 'doi', 'publication_year', 'document_type', 'language', 'keywords']
records = []
for doc in documents.to_dict('records'):
    out, flags, rejected, parse_status, model = resolve(doc)
    records.append({
        'document_id': doc['document_id'], 'file_name': doc['file_name'],
        'publinova_url': doc.get('publinova_url') or None,
        'page_count': int(doc['page_count']) if doc.get('page_count', '').isdigit() else None,
        'text_status': doc.get('text_status'),
        **{f: (out[f]['value'] if out[f]['value'] not in ([], '') else None) for f in FIELDS},
        'provenance': {f: {'source': out[f]['source'], 'confidence': out[f]['confidence']} for f in FIELDS},
        'review_status': 'ok' if not flags else 'needs_review',
        'review_flags': flags,
        'extraction': {'model': model, 'parse_status': parse_status, 'rejected_fields': rejected},
    })
for r in records:                                   # lists are never None in the API
    r['authors'] = r['authors'] or []
    r['keywords'] = r['keywords'] or []

final = pd.json_normalize(records)
print('Documents:', len(final))
display(final.review_status.value_counts().rename('documents').to_frame())

cov_rows = []
for f in FIELDS:
    filled = final[f].map(lambda v: bool(v) and v == v)
    cov_rows.append({'field': f, 'filled': int(filled.sum()), 'filled_%': round(100 * filled.mean(), 1),
                     **{c: int((final[f'provenance.{f}.confidence'] == c).sum()) for c in ('high', 'medium', 'low')}})
print('Coverage and confidence per field:')
display(pd.DataFrame(cov_rows))
print('Where the values come from:')
display(pd.DataFrame({f: final[f'provenance.{f}.source'].fillna('none').value_counts() for f in FIELDS}).fillna(0).astype(int))
rej = pd.Series([f for r in records for f in r['extraction']['rejected_fields']]).value_counts()
print('LLM values rejected by the checks (hallucination / mismatch indicator):')
display(rej.rename('documents').to_frame() if len(rej) else 'none')

Documents: 94


,documents
review_status,
needs_review,86
ok,8


Coverage and confidence per field:


,field,filled,filled_%,high,medium,low
0,title,38,40.4,6,32,0
1,authors,41,43.6,7,34,0
2,doi,33,35.1,2,31,0
3,publication_year,32,34.0,5,27,0
4,document_type,8,8.5,0,6,2
5,language,93,98.9,93,0,0
6,keywords,31,33.0,2,29,0


Where the values come from:


,title,authors,doi,publication_year,document_type,language,keywords
fasttext,0,0,0,0,0,87,0
llm,6,8,2,8,8,0,7
llm+fasttext,0,0,0,0,0,6,0
none,56,53,61,62,86,1,63
pdf_embedded,32,33,0,0,0,0,5
regex,0,0,31,24,0,0,19


LLM values rejected by the checks (hallucination / mismatch indicator):


,documents
keywords,5
language,2
title,2
authors,1


## 6. Evaluate against the gold standard

Compares the **raw LLM answer** (after cleaning only) with the **consolidated** result on the same documents. The gap between the two columns is what grounding and fallbacks add.

Only filled gold cells count, only for documents that have an LLM answer, and only for `EVAL_SPLITS`. `NONE` in the gold means "checked and absent".

In [14]:
GOLD_PATH = EXTRACT_DIR / 'gold_metadata.csv'
EVAL_THRESHOLD = {'title': 0.90, 'authors': 0.80, 'keywords': 0.80}

def gold_list(s):
    parts = [p.strip() for p in re.split(r'[;|]', s) if p.strip()]
    if len(parts) == 1 and ',' in parts[0]:
        parts = [p.strip() for p in parts[0].split(',') if p.strip()]
    return parts

def name_key(n):
    s = unicodedata.normalize('NFKD', str(n)).lower()
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return ' '.join(sorted(re.findall(r'[a-z0-9]+', s)))

def f1(g, p, key):
    g, p = {key(x) for x in g}, {key(x) for x in p}
    if not g and not p: return 1.0
    tp = len(g & p)
    if not tp: return 0.0
    pr, rc = tp / len(p), tp / len(g)
    return 2 * pr * rc / (pr + rc)

def score_field(f, truth, pred):
    truth = truth.strip()
    if truth.upper() == 'NONE':
        return float(not pred)
    if f == 'title':
        return difflib.SequenceMatcher(None, squash(truth), squash(pred or '')).ratio()
    if f == 'authors':
        return f1(gold_list(truth), pred or [], name_key)
    if f == 'keywords':
        return f1(gold_list(truth), pred or [], squash)
    if f == 'doi':
        return float(clean_doi(truth) == (pred or None))
    if f == 'publication_year':
        return float(clean_year(truth) == pred)
    if f == 'document_type':
        return float(norm_type(truth) == pred)
    if f == 'language':
        return float(norm_lang(truth) == pred)
    return float(squash(truth) == squash(pred or ''))

def raw_llm_values(d):
    llm, status, _ = llm_by_doc.get(d, ({}, 'no_llm_output', None))
    return {'title': norm_str(llm.get('title')),
            'authors': [n for n in (clean_author(x) for x in norm_list(llm.get('authors'))) if n],
            'doi': clean_doi(llm.get('doi')), 'publication_year': clean_year(llm.get('publication_year')),
            'document_type': norm_type(llm.get('document_type')), 'language': norm_lang(llm.get('language')),
            'keywords': norm_list(llm.get('keywords'))}

evaluation = pd.DataFrame()
if not GOLD_PATH.is_file():
    print('No gold_metadata.csv yet: run notebook 07 section 2 and verify the file by hand.')
else:
    gold = read_csv(GOLD_PATH)
    gold = gold[gold.split.isin(EVAL_SPLITS) & gold.document_id.isin(llm_by_doc)]
    by_doc = {r['document_id']: r for r in records}
    rows = []
    for g in gold.to_dict('records'):
        final_rec, raw_vals = by_doc.get(g['document_id']), raw_llm_values(g['document_id'])
        if not final_rec:
            continue
        for f in FIELDS:
            if not g[f].strip():
                continue
            for system, pred in (('raw_llm', raw_vals[f]), ('consolidated', final_rec[f])):
                s = score_field(f, g[f], pred)
                rows.append({'document_id': g['document_id'], 'file_name': g['file_name'], 'field': f, 'system': system,
                             'score': s, 'correct': s >= EVAL_THRESHOLD.get(f, 1.0), 'gold': g[f],
                             'predicted': '; '.join(pred) if isinstance(pred, list) else pred})
    evaluation = pd.DataFrame(rows)
    if evaluation.empty:
        print('Nothing to score: no verified gold cells for documents with an LLM answer.')
    else:
        evaluation.to_csv(API_DIR / 'evaluation.csv', index=False, encoding='utf-8-sig')
        summary = (evaluation.groupby(['field', 'system']).agg(n=('score', 'size'), accuracy=('correct', 'mean'),
                                                              mean_score=('score', 'mean')).round(2)
                   .unstack('system'))
        print(f'Scored {evaluation.document_id.nunique()} documents, splits {EVAL_SPLITS}')
        display(summary)

Scored 7 documents, splits ['dev']


n             accuracy           mean_score        
system   consolidated raw_llm consolidated raw_llm consolidated raw_llm
field                                                                  
authors             3       3         0.00    0.00         0.11    0.11
doi                 6       6         0.83    0.33         0.83    0.33
keywords            2       2         0.50    0.50         0.62    0.61
title               6       6         0.50    0.17         0.69    0.63

## 7. Review queue

Documents the pipeline is not sure about. Because the project relies on the LLM, you do not have to review them for the API to work; the flags and confidences are published so users can decide. This file is only a convenience for spot checks.

In [16]:
queue = final.loc[final.review_status == 'needs_review',
                  ['document_id', 'file_name', 'review_flags', 'extraction.parse_status', 'extraction.rejected_fields']]
queue.to_csv(API_DIR / 'review_queue.csv', index=False, encoding='utf-8-sig')
print(f'{len(queue)} of {len(final)} documents flagged; saved review_queue.csv')
display(queue.head(20))

86 of 94 documents flagged; saved review_queue.csv


,document_id,file_name,review_flags,extraction.parse_status,extraction.rejected_fields
1,doc_1f92b33dfeccc1c9,Business Case Vlissingen-checkpoint.pdf,"[no_llm_output, no_title, no_authors, no_publi...",no_llm_output,[]
2,doc_4d0489e9657d8e45,Nannochloropsis-checkpoint.pdf,[no_llm_output],no_llm_output,[]
3,doc_f5396bf79444d9ce,001_adaptive-capacities-for-diversified-flood-...,[no_llm_output],no_llm_output,[]
5,doc_571d8057e58bb19e,003_beach-user-perspectives-on-the-upscaling-o...,[no_llm_output],no_llm_output,[]
6,doc_5ebf988b95c44339,004_bonamia-free-flat-oyster-ostrea-edulis-l-s...,[no_llm_output],no_llm_output,[]
7,doc_5a6ef3b4929cdb64,005_building-relationships-in-higher-education...,"[no_llm_output, no_authors]",no_llm_output,[]
8,doc_4b983d71c002b202,006_defensive-versus-adaptive-action.pdf,[no_llm_output],no_llm_output,[]
9,doc_c7924ad1a4fb0d49,007_effect-of-ph-on-rhodomonas-salina-growth-b...,[no_llm_output],no_llm_output,[]
11,doc_4420f037ff3c4dd3,009_emotions-trust-and-expectations.pdf,[no_llm_output],no_llm_output,[]
13,doc_68b79731ca9eccbd,011_fine-tuning-the-flavor-of-tetraselmis-chui...,[no_llm_output],no_llm_output,[]


## 8. Build the SQLite database and exports

The database is written to a temporary file and swapped in with `os.replace`, so a running API never sees a half-written file. Empty values are stored as `NULL`. Authors and keywords are stored both as JSON (for responses) and as rows in their own tables (for filtering).

In [18]:
DB_PATH = API_DIR / 'metadata.db'
TMP_PATH = API_DIR / 'metadata.db.tmp'
if TMP_PATH.exists():
    TMP_PATH.unlink()

prov_cols = ',\n'.join(f'  {f}_source TEXT,\n  {f}_confidence TEXT' for f in FIELDS)
SCHEMA = f"""
CREATE TABLE documents (
  document_id TEXT PRIMARY KEY, file_name TEXT, publinova_url TEXT, page_count INTEGER, text_status TEXT,
  title TEXT, authors_json TEXT, doi TEXT, publication_year INTEGER, document_type TEXT, language TEXT,
  keywords_json TEXT,
{prov_cols},
  llm_model TEXT, llm_parse_status TEXT, llm_rejected_json TEXT,
  review_status TEXT NOT NULL, review_flags_json TEXT
);
CREATE TABLE authors  (document_id TEXT NOT NULL REFERENCES documents, position INTEGER, name TEXT NOT NULL, PRIMARY KEY (document_id, position));
CREATE TABLE keywords (document_id TEXT NOT NULL REFERENCES documents, position INTEGER, keyword TEXT NOT NULL, PRIMARY KEY (document_id, position));
CREATE TABLE meta (key TEXT PRIMARY KEY, value TEXT);
CREATE INDEX ix_doc_doi ON documents(doi);
CREATE INDEX ix_doc_year ON documents(publication_year);
CREATE INDEX ix_doc_lang ON documents(language);
CREATE INDEX ix_doc_type ON documents(document_type);
CREATE INDEX ix_authors_name ON authors(name COLLATE NOCASE);
CREATE INDEX ix_keywords_kw ON keywords(keyword COLLATE NOCASE);
"""

def db_row(r):
    row = {'document_id': r['document_id'], 'file_name': r['file_name'], 'publinova_url': r['publinova_url'],
           'page_count': r['page_count'], 'text_status': r['text_status'], 'title': r['title'],
           'authors_json': json.dumps(r['authors'], ensure_ascii=False),
           'doi': r['doi'], 'publication_year': r['publication_year'],
           'document_type': r['document_type'], 'language': r['language'],
           'keywords_json': json.dumps(r['keywords'], ensure_ascii=False),
           'llm_model': r['extraction']['model'], 'llm_parse_status': r['extraction']['parse_status'],
           'llm_rejected_json': json.dumps(r['extraction']['rejected_fields']),
           'review_status': r['review_status'], 'review_flags_json': json.dumps(r['review_flags'])}
    for f in FIELDS:
        row[f'{f}_source'] = r['provenance'][f]['source']
        row[f'{f}_confidence'] = r['provenance'][f]['confidence']
    return row

con = sqlite3.connect(TMP_PATH)
con.executescript(SCHEMA)
rows = [db_row(r) for r in records]
cols = list(rows[0])
con.executemany(f"INSERT INTO documents ({','.join(cols)}) VALUES ({','.join(':' + c for c in cols)})", rows)
con.executemany('INSERT INTO authors VALUES (?,?,?)',
                [(r['document_id'], i, n) for r in records for i, n in enumerate(r['authors'], 1)])
con.executemany('INSERT INTO keywords VALUES (?,?,?)',
                [(r['document_id'], i, k) for r in records for i, k in enumerate(r['keywords'], 1)])
models = sorted({r['extraction']['model'] for r in records if r['extraction']['model']})
meta = {'built_at': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'documents': str(len(records)), 'llm_models': ', '.join(models) or 'none',
        'documents_with_llm_output': str(sum(r['extraction']['parse_status'] != 'no_llm_output' for r in records)),
        'keep_ungrounded': str(KEEP_UNGROUNDED), 'schema_version': '1'}
con.executemany('INSERT INTO meta VALUES (?,?)', meta.items())
con.commit()
assert con.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
assert con.execute('SELECT COUNT(*) FROM documents').fetchone()[0] == len(records)
con.close()
os.replace(TMP_PATH, DB_PATH)

with (API_DIR / 'documents.jsonl').open('w', encoding='utf-8') as fh:
    for r in records:
        fh.write(json.dumps(r, ensure_ascii=False) + '\n')
flat = final.copy()
for col in ('authors', 'keywords', 'review_flags', 'extraction.rejected_fields'):
    flat[col] = flat[col].map(lambda v: '; '.join(v) if isinstance(v, list) else v)
flat.to_csv(API_DIR / 'documents.csv', index=False, encoding='utf-8-sig')

print('Built', DB_PATH, f'({DB_PATH.stat().st_size / 1024:.0f} KB)')
for k, v in meta.items():
    print(f'  {k}: {v}')

Built /Users/georgyfurs/Documents/ICT_Year4/CPS/DDD/output/api/metadata.db (104 KB)
  built_at: 2026-11-04T19:51:53+00:00
  documents: 94
  llm_models: Qwen/Qwen2.5-1.5B-Instruct
  documents_with_llm_output: 8
  keep_ungrounded: False
  schema_version: 1


## 9. Verify the saved database

Reads the file back, the way the API will, and compares it with the in-memory result.

In [20]:
chk = sqlite3.connect(f'{DB_PATH.as_uri()}?mode=ro', uri=True)
chk.row_factory = sqlite3.Row
n = chk.execute('SELECT COUNT(*) FROM documents').fetchone()[0]
assert n == len(records), 'row count differs from memory'
bad = chk.execute("SELECT COUNT(*) FROM documents WHERE review_status NOT IN ('ok','needs_review')").fetchone()[0]
assert bad == 0
orphans = chk.execute('SELECT COUNT(*) FROM authors WHERE document_id NOT IN (SELECT document_id FROM documents)').fetchone()[0]
assert orphans == 0
dup_doi = chk.execute("SELECT doi, COUNT(*) c FROM documents WHERE doi IS NOT NULL GROUP BY doi HAVING c > 1").fetchall()
print(f'{n} documents in the database, no orphans.')
if dup_doi:
    print('Repeated DOIs (same DOI on several files; check for duplicates or a cited DOI picked up by mistake):')
    display(pd.DataFrame([dict(r) for r in dup_doi]))
display(pd.DataFrame([dict(r) for r in chk.execute(
    'SELECT title, publication_year, doi, language, review_status FROM documents LIMIT 5')]))
chk.close()

94 documents in the database, no orphans.


,title,publication_year,doi,language,review_status
0,POLDER2C’s Animals on the Levee,2022.0,NaN,en,ok
1,NaN,NaN,NaN,en,needs_review
2,"The effect of drying, cell disruption and stor...",2023.0,10.1016/j.algal.2023.103092,en,needs_review
3,Adaptive Capacities for Diversified Flood Risk...,2019.0,10.3390/w11122643,en,needs_review
4,Social Psychology of Education,2019.0,10.1007/s11218-019-09488-4,en,ok



Next: notebook 09 tests and runs the API.
